# Explanations - Angles and internodes from tree graph

In [1]:
import networkx as nx
import numpy as np
import plotly.graph_objects as go

from plantdb.commons.test_database import test_database
from plantdb.commons.io import read_graph
from plantdb.commons.io import read_point_cloud

## Connect to the database & get the initial data

### Connect to the database

In [2]:
db = test_database('real_plant_analyzed', no_auth=True)
db.connect()

INFO     [test_database] File 'real_plant_analyzed.zip' exists locally. Skipping download.
INFO     [test_database] Verifying 'real_plant_analyzed.zip' MD5 hash value...
INFO     [LockManager] Starting cleanup of stale locks...
INFO     [LockManager] Cleaned up 0 stale lock files
WARNING  [UserManager] Empty user database file found under '/tmp/ROMI_DB_v06iotqm/users.json'
INFO     [UserManager] Creating guest user 'guest'
INFO     [UserManager] Creating admin user 'admin'
INFO     [UserManager] Welcome PlantDB Admin, please log in...'
Created admin user 'admin' with password 'admin'
Change it as soon as possible!
WARNING  [GroupManager] Empty groups database file found under '/tmp/ROMI_DB_v06iotqm/groups.json'


100%|██████████| 2/2 [00:00<00:00, 33.19scan/s]

INFO     [FSDB] Successfully connected to the database
ERROR    [FSDB] No username or token provided
WARNING  [metadata] Metadata key 'acquisition_date' was set to `None`!
INFO     [FSDB] Reloading scan 'real_plant_analyzed'...
INFO     [FSDB] Done!
INFO     [test_database] The test database is set up under '/tmp/ROMI_DB_v06iotqm'.
INFO     [LockManager] Starting cleanup of stale locks...
INFO     [LockManager] Cleaned up 0 stale lock files
WARNING  [GroupManager] Empty groups database file found under '/tmp/ROMI_DB_v06iotqm/groups.json'



100%|██████████| 2/2 [00:00<00:00, 62.66scan/s]

INFO     [FSDB] Successfully connected to the database


True

Once you are connected to the database, you can list the available scan *datasets* as follows:

In [3]:
db.list_scans()

[]

### Get the dataset

We now select the `real_plant_analyzed` dataset for the demo:

In [4]:
scan = db.get_scan("real_plant_analyzed")

Similarly, we can now list the available *filesets* as follows:

In [5]:
scan.list_filesets()

['images',
 'AnglesAndInternodes_1_0_2_0_6_0_6dd64fc595',
 'TreeGraph__False_CurveSkeleton_c304a2cc71',
 'CurveSkeleton__TriangleMesh_0393cb5708',
 'TriangleMesh_9_most_connected_t_open3d_00e095c359',
 'PointCloud_1_0_1_0_10_0_7ee836e5a9',
 'Voxels___x____300__450__colmap_camera_False_2a093f0ccc',
 'Masks_1__0__1__0____channel____rgb_5619aa428d',
 'Colmap_True_null_SIMPLE_RADIAL_ffcef49fdc',
 'Undistorted_SIMPLE_RADIAL_Colmap__a333f181b7']

### Get the fileset

We now select the `TreeGraph__False_CurveSkeleton_c304a2cc71` fileset for the demo:

In [6]:
fs = scan.get_fileset("TreeGraph__False_CurveSkeleton_c304a2cc71")

Similarly, we can now list the available *files* as follows:

In [7]:
fs.list_files()

['TreeGraph']

### Get the start file

We now select the `TreeGraph` file for the demo:

In [8]:
f = fs.get_file('TreeGraph')

You may have a look at the selected file path with:

In [9]:
f.path()

PosixPath('/tmp/ROMI_DB_v06iotqm/real_plant_analyzed/TreeGraph__False_CurveSkeleton_c304a2cc71/TreeGraph.p')

### Load the tree graph data

We use the `read_graph` reader from the `io` module of `plantdb`:

In [10]:
tree = read_graph(f)

We may now disconnect from the database as we will not need it anymore:

In [11]:
db.disconnect()

Let's have a look at the `tree` variable:

In [12]:
print(tree)

Graph with 948 nodes and 947 edges


In [13]:
tree.nodes[424]  # a fruit node

{'position': [408.8474948787428, 375.5788964944736, -63.3697262945477],
 'labels': ['stem'],
 'main_stem_id': 140}

In [14]:
tree.nodes[27]  # a stem node

{'position': [405.3074996426481, 378.4122600160056, -43.154360089921305],
 'labels': ['stem', 'node'],
 'fruit_id': 10,
 'main_stem_id': 163}

In [15]:
set(["_".join(tree.nodes[n]["labels"]) for n in tree.nodes])  # set of possible 'labels'

{'fruit', 'stem', 'stem_node'}

In [16]:
from plant3dvision.tree import get_root_node_id

root_node_id = get_root_node_id(tree)
root_node_id

265

In [17]:
from plant3dvision.tree import topological_distance

topo_dist = topological_distance(tree, root_node_id)

In [18]:
print({i:d for n, (i, d) in enumerate(topo_dist.items()) if n < 10})

{263: 1, np.int64(404): 2, 49: 3, np.int64(778): 4, 555: 5, 182: 6, np.int64(556): 7, 103: 8, np.int64(799): 9, 521: 10}


In [19]:
tree.nodes[300]  # confirm that node 300 has a 'main_stem_id' of 2 (distance)

{'position': [410.6460446202002, 371.92618802264775, -126.72703107027331],
 'labels': ['stem'],
 'main_stem_id': 74}

## View the original data

We use Plotly library to visualize the loaded triangular mesh.

Have a look at *Plotly Open Source Graphing Library for Python* here: https://plotly.com/python/

In [20]:
from plant3dvision.visu.plotly import plotly_treegraph

In [21]:
fig = plotly_treegraph(tree)
fig.show()

## Compute angles and internode length between successive organs (fruits)

We start by retreiving the list of `networkx.Graph` node ids that describe:
* the **main stem**: nodes that contains "stem" in their `label` attribute
* the **branching point**: nodes that contains "node" in their `label` attribute

In [22]:
from plant3dvision.arabidopsis import get_nodes_by_label

unordered_main_stem = get_nodes_by_label(tree, "stem")
unordered_branching_points = get_nodes_by_label(tree, "node")

They are said to be *unordered* as the node ids do not match the *natural order of organs* that are ordered from the bottom to the top of the plant.

### Create the ordered list of node ids for the *main stem* nodes

In [23]:
stem_dict = {}
for umn in unordered_main_stem:
    stem_dict[umn] = tree.nodes[umn]["main_stem_id"]
main_stem = [k for k, v in sorted(stem_dict.items(), key=lambda item: item[1])]

In [24]:
print(main_stem)

[265, 263, 404, 49, 778, 555, 182, 556, 103, 799, 521, 759, 259, 621, 653, 402, 333, 334, 697, 89, 90, 706, 91, 798, 88, 907, 926, 87, 911, 296, 884, 332, 719, 92, 403, 297, 0, 55, 336, 335, 264, 654, 337, 338, 339, 405, 631, 93, 406, 871, 879, 942, 938, 940, 266, 1, 594, 595, 727, 268, 728, 298, 369, 340, 946, 124, 513, 270, 598, 729, 562, 105, 106, 126, 300, 301, 127, 883, 525, 731, 130, 656, 5, 6, 780, 343, 131, 271, 657, 188, 189, 863, 344, 191, 190, 9, 659, 10, 847, 413, 937, 782, 660, 11, 12, 13, 902, 857, 848, 418, 18, 895, 108, 872, 515, 664, 60, 19, 767, 419, 192, 855, 609, 193, 20, 110, 785, 531, 140, 739, 305, 761, 276, 834, 277, 142, 111, 144, 143, 22, 424, 786, 787, 425, 921, 428, 97, 278, 601, 113, 724, 430, 622, 280, 197, 800, 672, 199, 673, 431, 537, 896, 859, 27, 638, 538, 612, 820, 150, 675, 541, 543, 151, 153, 813, 893, 156, 475, 31, 438, 378, 801, 47, 98, 200, 64, 440, 285, 160, 843, 348, 379, 161, 381, 639, 384, 317, 167, 822, 170, 172, 623, 854, 725, 174, 443, 641

### Create the ordered list of node ids for the *branching point* nodes

In [25]:
nodes_dict = {}
for ubp in unordered_branching_points:
    nodes_dict[ubp] = tree.nodes[ubp]["fruit_id"]
branching_points = [k for k, v in sorted(nodes_dict.items(), key=lambda item: item[1])]

In [26]:
print(branching_points)

[938, 946, 883, 131, 344, 902, 872, 60, 22, 428, 27, 813, 156, 379, 167, 869, 448, 480, 225, 459, 42, 585, 587, 243, 327, 291, 180, 80, 102, 933, 257, 401]


### Detect *weird* branching points

The canonical rule for *Arabidopsis thaliana* phyllotaxis dictate that, for a single branch (here our main stem), we should have a sucession of organs along the stem with **distinct branching points**.

We can check that we indeed follow this rule by looking at the **number of edges** attached to a branching point node.
The **connectivity degree** of each branching point should thus be equal to three: two *main stem edges* and one *fruit edge*.
This is done as follows:

In [27]:
weird_bp = [bp for bp in branching_points if tree.degree(bp) > 3]
print(weird_bp)

[938, 946, 291, 80, 102, 257]


Then we can have a look at the neighbors of those *weird branching points*, notably to their *nature* ("labels" node attibute), to gain insight on what is happening there.

This can be done as follows:

In [28]:
wbp_edge_labels = {}
for wbp in weird_bp:
    neighbors = list(tree.neighbors(wbp))
    wbp_edge_labels[wbp] = [tree.nodes[nei]['labels'] for nei in neighbors]

In [29]:
wbp_edge_labels

{938: [['fruit'], ['fruit'], ['stem'], ['stem']],
 946: [['stem'], ['stem'], ['fruit'], ['fruit']],
 291: [['fruit'], ['stem'], ['stem'], ['fruit']],
 80: [['fruit'], ['stem'], ['fruit'], ['stem', 'node']],
 102: [['stem', 'node'], ['fruit'], ['stem', 'node'], ['fruit']],
 257: [['stem'], ['fruit'], ['fruit'], ['stem', 'node']]}

We can see some branching points with **two fruits attached**!

As an example, let's see how we can differentiate the two fruits using `connected_components` function from `networkx`.

In [30]:
from plant3dvision.arabidopsis import get_fruit

bp_node_id = 291  # select a branching point node with two fruits attached
# Get the fruit id corresponding to the branching point node:
fruit_id = [tree.nodes[nei]['fruit_id'] for nei in tree.neighbors(bp_node_id) if "fruit" in tree.nodes[nei]['labels']][
    0]
# Get the list of node fruit attached to this branching point node:
fruit_nodes = get_fruit(tree, fruit_id)
# Get the subgraph with only fruit nodes:
fruit_tree = tree.subgraph(fruit_nodes)
# Get the list of connected components:
fruit_nodes = list(nx.connected_components(fruit_tree))

In [31]:
print(fruit_nodes)

[{391, np.int64(472), np.int64(649), np.int64(714), 75, np.int64(715), np.int64(845), 392, 503, 504, np.int64(473), 255}, {np.int64(870), np.int64(711), np.int64(712), np.int64(713), 73, 72, 74, 877, np.int64(878), 178, 179, np.int64(502), 824, np.int64(825), 251, 252, np.int64(253), np.int64(254)}]


## Case study 1 - A regular branching point

Let's start with a "simple" case where the branching point:
* has a single fruit attached
* is well separated from the previous and next branching points

In [32]:
bp_node_id = 291  # manually selected after exploration of the "Tree graph" figure above

In [33]:
# Get the index of the selected node in the ordered branching points list
bp_idx = branching_points.index(bp_node_id)

In [34]:
# Get the node ids of the previous and next branching points from the ordered branching points list
prev_bp_node_id = branching_points[bp_idx - 1]
next_bp_node_id = branching_points[bp_idx + 1]

In [35]:
print(f"Selected branching point has node id {bp_node_id}.")
print(f"Previous branching point has node id {prev_bp_node_id}.")
print(f"Next branching point has node id {next_bp_node_id}.")

Selected branching point has node id 291.
Previous branching point has node id 327.
Next branching point has node id 180.


### Distance to previous and next branching points

In [36]:
from scipy.spatial.distance import euclidean

In [37]:
bp_coord = np.array(tree.nodes[bp_node_id]["position"])  # get the branching point coordinates

In [38]:
distances = []
for bp_id in [prev_bp_node_id, next_bp_node_id]:
    bp_c = np.array(tree.nodes[bp_id]["position"])
    distances.append(euclidean(bp_coord, bp_c))

In [39]:
distances

[9.365442556567228, 4.716480960593531]

### Select stem nodes between surrounding branching points

It is now possible to **list all stem node ids** between the two branching points surrounding the one we selected:

In [40]:
prev_bp_stem_idx = main_stem.index(prev_bp_node_id)
next_bp_stem_idx = main_stem.index(next_bp_node_id)

In [41]:
stem_nodes = main_stem[prev_bp_stem_idx + 1:next_bp_stem_idx]
print(stem_nodes)

[389, 245, 591, 592, 247, 248, 249, 328, 119, 292, 291, 864, 329, 839]


Note that this **exclude** the surrounding branching points!

### Select fruit nodes attached to branching point

We now search the *fruit id* attached to the selected branching point as follows:

In [42]:
fruit_id = [tree.nodes[nei]['fruit_id'] for nei in tree.neighbors(bp_node_id) if "fruit" in tree.nodes[nei]['labels']][
    0]
fruit_id

25

We can recover the **list of fruit node ids** corresponding to the *fruit id* attached to the selected branching point using the `get_fruit` function from the `plant3dvision.arabidopsis` module as follows:

In [43]:
from plant3dvision.arabidopsis import get_fruit

fruit_nodes = get_fruit(tree, fruit_id)
print(fruit_nodes)

[72, 73, 74, 75, 178, 179, 251, 252, 253, 254, 255, 391, 392, 472, 473, 502, 503, 504, 649, 711, 712, 713, 714, 715, 824, 825, 845, 870, 877, 878]


Again, these nodes are NOT sorted!

Then we **extract the subgraph** corresponding to those selected fruit & stem nodes:

In [44]:
import networkx as nx

subtree = nx.subgraph(tree, fruit_nodes + stem_nodes)

Let's represent this subgraph:

In [45]:
fig = plotly_treegraph(subtree, height=500, title="SubTree graph", mode="lines+markers", marker_kwargs={'size':2})
fig.show()

We want to estimate the fruit and stem directions by means of vectors to later be able to estimate the angles between successive fruits.

In [46]:
bp_coord = np.array(subtree.nodes[bp_node_id]["position"])  # get the branching point coordinates

### Get the stem point 3D coordinates between surrounding branching points

Let's start by using all selected *stem nodes* in the sub-tree (all stem nodes between the two branching points surrounding the selected one).
We also **exclude the branching point node** as its position might be **inaccuratly estimated** by the skeletonization algorithm (zig-zag pattern at the branching point).

In [47]:
stem_points = np.array([subtree.nodes[n]["position"] for n in stem_nodes if n != bp_node_id])

We sort the *stem points* by their z-axis coordinates, from lowest to highest points as follows:

In [48]:
stem_points = np.array([k for k in sorted(stem_points, key=lambda item: item[2])])

In [49]:
stem_mean = stem_points.mean(axis=0)

### Get the fruit point 3D coordinates by distance to branching point

To get an accurate estmation at the branching point, we may not want to use all *fruit nodes* coordinates as the fruit may be curvy and thus "stay close" to the branching point.
This will also provide us a more **consistent direction estimations** for all fruits. 

We thus limit the sampling by selecting *fruit nodes* **within a given distance to the branching point** by setting a `max_fruit_node_dist` variable.

In [50]:
max_fruit_node_dist = 10  # same unit as coordinates, so in millimeters!

In [51]:
from scipy.spatial.distance import euclidean

# Get all fruit node coordinates:
fruit_points = [subtree.nodes[n]["position"] for n in fruit_nodes]

In [52]:
# Restrict to those within a given Euclidean distance to branching point:
fruit_points = [fp for fp in fruit_points if euclidean(bp_coord, fp) <= max_fruit_node_dist]

In [53]:
{(prev, next): euclidean(tree.nodes[prev]["position"], tree.nodes[next]["position"]) for prev, next in
 zip(fruit_nodes[:-1], fruit_nodes[1:])}

{(72, 73): 3.677283617065049,
 (73, 74): 12.100546792429666,
 (74, 75): 21.642007837107972,
 (75, 178): 13.968553171537064,
 (178, 179): 2.2260776196294128,
 (179, 251): 7.335291956339239,
 (251, 252): 3.2011052776599302,
 (252, 253): 3.299600868093371,
 (253, 254): 2.2480216420903303,
 (254, 255): 19.178056973332396,
 (255, 391): 6.3076699789848885,
 (391, 392): 10.948302248336898,
 (392, 472): 3.6297253004908474,
 (472, 473): 1.433923910083008,
 (473, 502): 15.626215131856057,
 (502, 503): 10.576324992526056,
 (503, 504): 1.768381279560727,
 (504, 649): 4.725916236655272,
 (649, 711): 15.599635014295988,
 (711, 712): 0.7147299408907124,
 (712, 713): 3.7406567690771744,
 (713, 714): 11.793305400128409,
 (714, 715): 1.6829433112683763,
 (715, 824): 22.99976863959716,
 (824, 825): 0.5450573622913434,
 (825, 845): 27.692521378290007,
 (845, 870): 29.00992766313383,
 (870, 877): 3.8231183862792277,
 (877, 878): 0.8126467655454804}

### Project all points in the best 2D plane

In [54]:
plane_points = np.array(list(fruit_points) + list(stem_points))
c_plane_points = plane_points - stem_mean

In [55]:
# -- Singular Value Decomposition (SVD) of centered coordinates:
U, D, V = np.linalg.svd(c_plane_points)
V = V.T
# -- Projection matrix to 2D plane:
proj_mat = np.dot(V[:, 0:2], V[:, 0:2].T)

In [56]:
proj_fruit_points = np.dot(fruit_points - stem_mean, proj_mat) + stem_mean
proj_stem_points = np.dot(stem_points - stem_mean, proj_mat) + stem_mean

### Estimate the stem direction

Let's **find the best fitting line** to the sampled *stem nodes* coordinates centered on their mean using SVD:

In [57]:
uu, dd, vv = np.linalg.svd(proj_stem_points - stem_mean)

In [58]:
stem_dir = vv[0]
stem_dir

array([-0.14757565, -0.27436444,  0.95023449])

### Estimate the fruit direction

Let's **find the best fitting line** to the sampled *fruit nodes* coordinates passing trought the branching point using SVD:

In [59]:
fruit_points = np.array(proj_fruit_points)
uu, dd, vv = np.linalg.svd(proj_fruit_points - stem_mean)

In [60]:
fruit_dir = vv[0]
# Check if the first fruit point is higher than mean stem point and reverse
if (abs(proj_fruit_points[0]) - abs(stem_mean))[-1] < 0:
    fruit_dir = -fruit_dir
fruit_dir

array([-0.7074579 ,  0.55249899,  0.44073595])

### Visualize the estimated directions

In [61]:
fig = plotly_treegraph(subtree, height=700,
                       title="SubTree graph with estimated fruit (red) and stem (blue) directions.")

# Add selected fruit nodes as red points:
x, y, z = fruit_points.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers", name="Selected fruit nodes",
                  marker=dict(color='red', size=2, opacity=1))
# Add best fit for projected fruit nodes as red line:
linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Fruit direction",
                  marker=dict(color='red', size=3, opacity=0.8, symbol="diamond"))
# Add estimated fruit vector as red cone:
x, y, z = stem_mean.T  # cone origin is located at stem mean point
u, v, w = fruit_dir.T
fig.add_cone(x=[x], y=[y], z=[z], u=[u], v=[v], w=[w], name="Fruit direction",
             sizemode="scaled", sizeref=10, showscale=False, opacity=0.4)

# Add selected stem nodes as blue points:
x, y, z = stem_points.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers", name="Selected stem nodes",
                  marker=dict(color='blue', size=2, opacity=0.8))
# Add best fit for projected stem nodes as blue line:
mini = euclidean(stem_mean, stem_points[0])  # distance to lowest stem point
maxi = euclidean(stem_mean, stem_points[-1])  # distance to highest stem poin
linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Stem direction",
                  marker=dict(color='blue', size=3, opacity=0.8, symbol="diamond"))

### Visualize the estimated direction with the point-cloud

In [62]:
# Get the point cloud:
db.connect()
scan = db.get_scan("real_plant_analyzed")
fs = scan.get_fileset("PointCloud_1_0_1_0_10_0_7ee836e5a9")
f = fs.get_file('PointCloud')
pcd = read_point_cloud(f)
db.disconnect()

100%|██████████| 2/2 [00:00<00:00, 38.94scan/s]

INFO     [FSDB] Successfully connected to the database


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [63]:
# Compute the bounding box to crop point cloud in Z:
bbox = {"x": [], "y": [], "z": []}
bbox["z"] = [np.min(stem_points, axis=0)[2], np.max(stem_points, axis=0)[2]]
bbox["x"] = [np.min(pcd.points, axis=0)[0], np.max(pcd.points, axis=0)[0]]
bbox["y"] = [np.min(pcd.points, axis=0)[1], np.max(pcd.points, axis=0)[1]]
bbox

{'x': [np.float64(349.66480713675526), np.float64(428.83826778597415)],
 'y': [np.float64(343.68261495482255), np.float64(402.32510263716915)],
 'z': [np.float64(70.0782144929474), np.float64(80.20490709467151)]}

In [64]:
# Crop the point cloud:
from plant3dvision.proc3d import crop_point_cloud

pcd = crop_point_cloud(pcd, bbox)

In [65]:
x, y, z = np.array(pcd.points).T
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode="markers", name="Point cloud",
                                   marker=dict(size=1, color='green', opacity=0.8))])

# Add best fit line as red points:
linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Fruit direction",
                  marker=dict(color='red', size=3, opacity=0.8, symbol="diamond"))

# Add best fit line as blue points:
mini = euclidean(stem_mean, stem_points[0])  # distance to lowest stem point
maxi = euclidean(stem_mean, stem_points[-1])  # distance to highest stem poin
linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Stem direction",
                  marker=dict(color='blue', size=3, opacity=0.8, symbol="diamond"))

fig.update_layout(height=600, title="Point cloud with estimated fruit (red) and stem (blue) directions.",
                  showlegend=True)
fig.update_scenes(aspectmode='data')

fig.show()

## Case study 2 - A branching point with short internode distance

Now we consider a more tricky case where the branching point:
* has a single fruit attached
* is close the previous and/or next branching points

In [66]:
bp_node_id = 938  # manually selected after exploration of the "Tree graph" figure above

In [67]:
bp_idx = branching_points.index(bp_node_id)

In [68]:
prev_bp_node_id = branching_points[bp_idx - 1]
next_bp_node_id = branching_points[bp_idx + 1]

In [69]:
print(f"Selected branching point has node id {bp_node_id}.")
print(f"Previous branching point has node id {prev_bp_node_id}.")
print(f"Next branching point has node id {next_bp_node_id}.")

Selected branching point has node id 938.
Previous branching point has node id 401.
Next branching point has node id 946.


### Distance to previous and next branching points

In [70]:
from scipy.spatial.distance import euclidean

In [71]:
bp_coord = np.array(tree.nodes[bp_node_id]["position"])  # get the branching point coordinates

In [72]:
distances = []
for bp_nid in [prev_bp_node_id, next_bp_node_id]:
    bp_c = np.array(tree.nodes[bp_nid]["position"])
    distances.append(euclidean(bp_coord, bp_c))

In [73]:
distances

[245.6611803362897, 13.172659386382053]

As we can see, both are below 10mm, the distance to branching point we previously used to select the *fruit nodes*.

### Select stem nodes by distance to branching point

It is now possible to **list all stem node ids** between the two branching points surrounding the one we selected:

In [74]:
bp_coord = np.array(tree.nodes[bp_node_id]["position"])  # get the branching point coordinates

In [75]:
bp_stem_idx = main_stem.index(bp_node_id)  # get the branching point index in the ordered stem nodes list

In [76]:
max_stem_node_dist = 10  # same unit as coordinates, so in millimeters!
stem_nodes = {bp_node_id}  # start the set of selected stem nodes with the branching point node id

In [77]:
# Forward search towards next branching point:
for node in main_stem[bp_stem_idx:]:
    node_coord = np.array(tree.nodes[node]["position"])  # get the current node coordinates
    if euclidean(bp_coord, node_coord) > max_stem_node_dist:
        break
    stem_nodes |= {node}

In [78]:
# Backward search towards previous branching point:
bp_stem_idx = main_stem.index(bp_node_id)
for node in main_stem[:bp_stem_idx][::-1]:
    node_coord = np.array(tree.nodes[node]["position"])  # get the current node coordinates
    if euclidean(bp_coord, node_coord) > max_stem_node_dist:
        break
    stem_nodes |= {node}

In [79]:
stem_nodes = list(stem_nodes)
print(stem_nodes)

[1, 871, 631, 938, 266, 268, 940, 298, 942, 879, 594, 595, 339, 405, 406, 727, 728, 93]


In [80]:
[n in stem_nodes for n in [bp_node_id, prev_bp_node_id, next_bp_node_id]]

[True, False, False]

Note that this **include** the selected, previous and next branching points!

### Select fruit nodes attached to branching point

We now search the *fruit id* attached to the selected branching point as follows:

In [81]:
fruit_id = [tree.nodes[nei]['fruit_id'] for nei in tree.neighbors(bp_node_id) if "fruit" in tree.nodes[nei]['labels']][
    0]
fruit_id

0

We can recover the **list of fruit node ids** corresponding to the *fruit id* attached to the selected branching point using the `get_fruit` function from the `plant3dvision.arabidopsis` module as follows:

In [82]:
from plant3dvision.arabidopsis import get_fruit

fruit_nodes = get_fruit(tree, fruit_id)
print(fruit_nodes)

[50, 51, 52, 53, 54, 104, 121, 122, 123, 125, 183, 184, 187, 260, 261, 269, 490, 523, 524, 557, 558, 560, 596, 597, 779, 816, 852, 865, 866, 867, 868, 897, 908, 909, 912, 913, 923, 943, 945]


Again, these nodes are NOT sorted!

Then we **extract the subgraph** corresponding to those selected fruit & stem nodes:

In [83]:
import networkx as nx

subtree = nx.subgraph(tree, fruit_nodes + stem_nodes)

Let's represent this subgraph:

In [84]:
fig = plotly_treegraph(subtree, height=500, title="SubTree graph")
fig.show()

We want to estimate the fruit and stem directions by means of vectors to later be able to estimate the angles between successive fruits.

In [85]:
bp_coord = np.array(subtree.nodes[bp_node_id]["position"])  # get the branching point coordinates

### Get the stem point 3D coordinates

Let's start by using all selected *stem nodes* in the sub-tree (all stem nodes between the two branching points surrounding the selected one).
We also **exclude the branching point node** as its position might be **inaccuratly estimated** by the skeletonization algorithm (zig-zag pattern at the branching point).

In [86]:
stem_points = np.array([subtree.nodes[n]["position"] for n in stem_nodes if "node" not in subtree.nodes[n]["labels"]])

We sort the *stem points* by their z-axis coordinates, from lowest to highest points as follows:

In [87]:
stem_points = np.array([k for k in sorted(stem_points, key=lambda item: item[2])])

In [88]:
stem_mean = stem_points.mean(axis=0)

### Get the fruit point 3D coordinates

To get an accurate estmation at the branching point, we may not want to use all *fruit nodes* coordinates as the fruit may be curvy and thus "stay close" to the branching point.
This will also provide us a more **consistent direction estimations** for all fruits. 

We thus limit the sampling by selecting *fruit nodes* **within a given distance to the branching point** by setting a `max_fruit_node_dist` variable.

In [89]:
max_fruit_node_dist = 10  # same unit as coordinates, so in millimeters!

In [90]:
from scipy.spatial.distance import euclidean

# Get all fruit node coordinates:
fruit_points = [subtree.nodes[n]["position"] for n in fruit_nodes]
# Restrict to those within a given Euclidean distance to branching point:
fruit_points = [fp for fp in fruit_points if euclidean(bp_coord, fp) <= max_fruit_node_dist]

### Project all points in the best 2D plane

In [91]:
plane_points = np.array(list(fruit_points) + list(stem_points))
c_plane_points = plane_points - stem_mean

In [92]:
# -- Singular Value Decomposition (SVD) of centered coordinates:
U, D, V = np.linalg.svd(c_plane_points)
V = V.T
# -- Projection matrix to 2D plane:
proj_mat = np.dot(V[:, 0:2], V[:, 0:2].T)

In [93]:
proj_fruit_points = np.dot(fruit_points - stem_mean, proj_mat) + stem_mean
proj_stem_points = np.dot(stem_points - stem_mean, proj_mat) + stem_mean

### Estimate the stem direction

Let's **find the best fitting line** to the sampled *stem nodes* coordinates centered on their mean using SVD:

In [94]:
uu, dd, vv = np.linalg.svd(proj_stem_points - stem_mean)

In [95]:
stem_dir = vv[0]
stem_dir

array([-0.02734219,  0.0644984 ,  0.99754316])

### Estimate the fruit direction

Let's **find the best fitting line** to the sampled *fruit nodes* coordinates passing trought the branching point using SVD:

In [96]:
fruit_points = np.array(proj_fruit_points)
uu, dd, vv = np.linalg.svd(proj_fruit_points - stem_mean)

In [97]:
fruit_dir = vv[0]
# Check if the first fruit point is higher than mean stem point and reverse
if (abs(proj_fruit_points[0]) - abs(stem_mean))[-1] < 0:
    fruit_dir = -fruit_dir
fruit_dir

array([0.80840334, 0.58430297, 0.07123261])

### Visualize the estimated directions

In [98]:
fig = plotly_treegraph(subtree, height=600,
                       title="SubTree graph with estimated fruit (red) and stem (blue) directions.")

# Add selected fruit nodes as red points:
x, y, z = fruit_points.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers", name="Selected fruit nodes",
                  marker=dict(color='red', size=2, opacity=1))
# Add best fit for projected fruit nodes as red line:
linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Fruit direction",
                  marker=dict(color='red', size=3, opacity=0.8, symbol="diamond"))
# Add estimated fruit vector as red cone:
x, y, z = stem_mean.T  # cone origin is located at stem mean point
u, v, w = fruit_dir.T
fig.add_cone(x=[x], y=[y], z=[z], u=[u], v=[v], w=[w], name="Fruit direction",
             sizemode="scaled", sizeref=10, showscale=False, opacity=0.4)

# Add selected stem nodes as blue points:
x, y, z = stem_points.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers", name="Selected stem nodes",
                  marker=dict(color='blue', size=2, opacity=0.8))
# Add best fit for projected stem nodes as blue line:
mini = euclidean(stem_mean, stem_points[0])  # distance to the lowest stem point
maxi = euclidean(stem_mean, stem_points[-1])  # distance to the highest stem point
linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Stem direction",
                  marker=dict(color='blue', size=3, opacity=0.8, symbol="diamond"))

### Visualize the estimated direction with the point-cloud

In [99]:
# Get the point cloud:
db.connect()
scan = db.get_scan("real_plant_analyzed")
fs = scan.get_fileset("PointCloud_1_0_1_0_10_0_7ee836e5a9")
f = fs.get_file('PointCloud')
pcd = read_point_cloud(f)
db.disconnect()

100%|██████████| 2/2 [00:00<00:00, 49.67scan/s]

INFO     [FSDB] Successfully connected to the database


In [100]:
# Compute the bounding box to crop point cloud in Z:
bbox = {"x": [], "y": [], "z": []}
bbox["z"] = [np.min(stem_points, axis=0)[2], np.max(stem_points, axis=0)[2]]
bbox["x"] = [np.min(pcd.points, axis=0)[0], np.max(pcd.points, axis=0)[0]]
bbox["y"] = [np.min(pcd.points, axis=0)[1], np.max(pcd.points, axis=0)[1]]
bbox

{'x': [np.float64(349.66480713675526), np.float64(428.83826778597415)],
 'y': [np.float64(343.68261495482255), np.float64(402.32510263716915)],
 'z': [np.float64(-156.13822983877537), np.float64(-138.92813513953362)]}

In [101]:
# Crop the point cloud:
from plant3dvision.proc3d import crop_point_cloud

pcd = crop_point_cloud(pcd, bbox)

In [102]:
x, y, z = np.array(pcd.points).T
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode="markers", name="Point cloud",
                                   marker=dict(size=1, color='green', opacity=0.8))])

# Add best fit line as red points:
linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Fruit direction",
                  marker=dict(color='red', size=3, opacity=0.8, symbol="diamond"))

# Add best fit line as blue points:
mini = euclidean(stem_mean, stem_points[0])  # distance to the lowest stem point
maxi = euclidean(stem_mean, stem_points[-1])  # distance to the highest stem point
linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + stem_mean
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines",name="Stem direction",
                  marker=dict(color='blue', size=3, opacity=0.8, symbol="diamond"))

fig.update_layout(height=600, title="Point cloud with estimated fruit (red) and stem (blue) directions.",
                  showlegend=True)
fig.update_scenes(aspectmode='data')

fig.show()

## Generalization - Stem and fruit nodes sampling

We use the following rules to sample the fruits and stem nodes:
* *stem points* are sampled around the branching point within a given Euclidean distance from it
* *fruits points* are sample from the branching point to a maximum Euclidean distance from it
* if multiple fruits are attached to the same branching point, they are treated individually

## Project the branching point on the stem nodes line

It may be preferable to **project the branching point node** on the line formed by the *stem nodes* rather than taking their mean coordinate...

Let's see how that goes!

In [103]:
from scipy.spatial.distance import euclidean

from plant3dvision.tree import nodes_coordinates
from plant3dvision.tree import select_stem_nodes_by_euclidean_distance
from plant3dvision.tree import select_fruit_nodes
from plant3dvision.arabidopsis import get_proj_matrix
from plant3dvision.arabidopsis import project_points
from plant3dvision.arabidopsis import vector_from_points

In [104]:
bp_node_id = 127

In [105]:
bp_coord = nodes_coordinates(tree, [bp_node_id])[0]  # get the branching point coordinates

In [106]:
stem_nodes = select_stem_nodes_by_euclidean_distance(tree, bp_node_id, max_node_dist=10.)
stem_points = nodes_coordinates(tree, stem_nodes)
stem_mean = stem_points.mean(axis=0)

In [107]:
stem_line_proj_mat = get_proj_matrix(stem_points, dim=1)
stem_line_proj_mat  # a 3x3 projection matrix

array([[ 2.35598620e-09, -1.33370073e-06, -4.85201755e-05],
       [-1.33370073e-06,  7.54994931e-04,  2.74667965e-02],
       [-4.85201755e-05,  2.74667965e-02,  9.99245003e-01]])

In [108]:
proj_bp_coord = project_points(bp_coord, stem_line_proj_mat, stem_mean)
bp_coord, proj_bp_coord

(array([ 410.88198919,  371.86379244, -125.51171183]),
 array([ 410.63935735,  372.12599813, -125.518931  ]))

In [109]:
fruit_nodes = select_fruit_nodes(tree, bp_node_id, max_node_dist=None)
subtree = tree.subgraph(stem_nodes + fruit_nodes[0])
stem_dir = vector_from_points(stem_points, origin=proj_bp_coord)

IndexError: list index out of range

In [ ]:
fig = plotly_treegraph(subtree, height=600,
                       title="SubTree graph with estimated stem (blue) direction, branching point (red) and projected branching point (green).")
# Add best fit for projected stem nodes as blue line:
mini = euclidean(stem_mean, stem_points[0])  # distance to the lowest stem point
maxi = euclidean(stem_mean, stem_points[-1])  # distance to the highest stem point
linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + proj_bp_coord
x, y, z = linepts.T
fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Stem direction",
                  marker=dict(color='blue', size=3, opacity=0.8, symbol="diamond"))

# Add projected branching point as green point:
x, y, z = proj_bp_coord
fig.add_scatter3d(x=[x], y=[y], z=[z], mode="markers", name="Projected branching point",
                  marker=dict(color='green', size=3, opacity=1, symbol="x"))

# Add stem mean point as red point:
x, y, z = stem_mean
fig.add_scatter3d(x=[x], y=[y], z=[z], mode="markers", name="Stem mean point",
                  marker=dict(color='red', size=3, opacity=1, symbol="x"))

We can compute the distance between the *original branching point* to the *projected branching point* as follows:

In [ ]:
euclidean(bp_coord, proj_bp_coord)

We can also compute the distance between the *mean stem point* to the *projected branching point* as follows:

In [ ]:
euclidean(stem_mean, proj_bp_coord)

## Generalization - Stem and fruit direction estimation

We use the following rules to estimate the fruits and stem directions:
* *stem points* are sampled around the branching point within a given path Euclidean distance from it
* *fruits points* are sample from the branching point to a maximum path Euclidean distance from it
* if multiple fruits are attached to the same branching point, they are treated individually
* the *branching point* is projected on the stem line

In [ ]:
from plant3dvision.arabidopsis import get_nodes_by_label

unordered_branching_points = get_nodes_by_label(tree, "node")

nodes_dict = {}
for ubp in unordered_branching_points:
    nodes_dict[ubp] = tree.nodes[ubp]["fruit_id"]
branching_points = [k for k, v in sorted(nodes_dict.items(), key=lambda item: item[1])]

In [ ]:
max_node_dist = 10.
fruit_dirs = []
stem_dirs = []
bp_coords = []

for bp_node_id in branching_points:
    bp_coord = nodes_coordinates(tree, [bp_node_id])[0]  # get the branching point coordinates
    # Get the lists of fruit and stem nodes:
    fruit_nodes_list = select_fruit_nodes(tree, bp_node_id, max_node_dist)
    stem_nodes = select_stem_nodes_by_euclidean_distance(tree, bp_node_id, max_node_dist)
    # Get the coordinates of selected fruit and stem points:
    stem_points = nodes_coordinates(tree, stem_nodes)
    # Compute stem line projection matrix:
    stem_line_proj_mat = get_proj_matrix(stem_points, dim=1)
    # Project stem points on stem line:
    proj_stem_points = project_points(stem_points, stem_line_proj_mat)
    # Project branching point on stem line:
    proj_stem_mean = proj_stem_points.mean(axis=0)
    proj_bp_coord = project_points(bp_coord, stem_line_proj_mat, proj_stem_mean)
    # Compute stem direction:
    stem_dir = vector_from_points(proj_stem_points, origin=proj_bp_coord)
    if stem_dir.dot(proj_stem_points[-1,:] - proj_bp_coord):
        stem_dir = -stem_dir

    for fruit_nodes in fruit_nodes_list:
        # Get the coordinates of selected fruit nodes:
        fruit_points = nodes_coordinates(tree, fruit_nodes)
        # Compute fruit direction:
        fruit_dir = vector_from_points(fruit_points, origin=proj_bp_coord)
        if fruit_dir.dot(fruit_points[-1,:] - proj_bp_coord):
            fruit_dir = -fruit_dir
        fruit_dirs.append(fruit_dir)
        stem_dirs.append(stem_dir)
        bp_coords.append(proj_bp_coord)

In [ ]:
# Get the point cloud:
db.connect()
scan = db.get_scan("real_plant_analyzed")
fs = scan.get_fileset("PointCloud_1_0_1_0_10_0_7ee836e5a9")
f = fs.get_file('PointCloud')
pcd = read_point_cloud(f)
db.disconnect()

In [ ]:
rng = np.random.default_rng()
ds_pcd = rng.choice(pcd.points, 10000)

In [ ]:
x, y, z = ds_pcd.T
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode="markers", name="Point cloud",
                                   marker=dict(size=1, color='green', opacity=0.5))])

for n, fruit_dir in enumerate(fruit_dirs):
    # Add best fit line as red points:
    linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + bp_coords[n]
    x, y, z = linepts.T
    fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name=f"fruit {n}",
                      marker=dict(size=3, opacity=0.8, symbol="diamond"))

prev_bp = [0., 0., 0.]
for n, stem_dir in enumerate(stem_dirs):
    if np.sum(bp_coords[n] - prev_bp) == 0.:
        continue  # skip if branching point is the same (to avoid duplicates)
    else:
        prev_bp = bp_coords[n]
    # Add best fit line as blue points:
    linepts = stem_dir * np.mgrid[-10:10:2j][:, np.newaxis] + bp_coords[n]
    x, y, z = linepts.T
    fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name=f"stem {n}",
                      marker=dict(color='blue', size=3, opacity=0.8, symbol="diamond"))

fig.update_layout(height=1200, title="Point cloud with estimated fruit (red) and stem (blue) directions.",
                  showlegend=True)
fig.update_scenes(aspectmode='data')

fig.show()